## Introduction

The goal of this notebook is to **breakdown financing P&L**

## Import

In [118]:
import numpy as np
import cvxpy as cp
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import itertools as it
import plotly.express as px
from sklearn import linear_model
from python_module.pricing_model import BSMModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.2f}'.format

## Custom function

In [119]:
def compute_box_pv(F, T, r, K1=90, K2=110, sigma=0.2):
    a = BSMModel.compute_option_with_forward(F=F, K=K1, T=T, r=r, sigma=sigma, option_type='call')
    b = BSMModel.compute_option_with_forward(F=F, K=K2, T=T, r=r, sigma=sigma, option_type='call')
    c = BSMModel.compute_option_with_forward(F=F, K=K1, T=T, r=r, sigma=sigma, option_type='put')
    d = BSMModel.compute_option_with_forward(F=F, K=K2, T=T, r=r, sigma=sigma, option_type='put')
    q = 100/(K2-K1)
    return (a-b-c+d)*q

## Inputs

In [120]:
r = 0.1
nb_days = 252

## BOX Analysis

In [121]:
# Sanity check compute_box_pv function
for T in [1, 0]:
    for F in [50, 90, 100, 110, 150]:
        pv = compute_box_pv(F=F, T=T, r=r)
        print(f"F={F}, T={T}, PV={pv}")
r_box = np.log(compute_box_pv(F=F, T=0, r=r)/compute_box_pv(F=F, T=1, r=r))
print(f"r={r}, implied box rate = {r_box}")

F=50, T=1, PV=90.48374180359595
F=90, T=1, PV=90.483741803596
F=100, T=1, PV=90.48374180359602
F=110, T=1, PV=90.48374180359595
F=150, T=1, PV=90.4837418035959
F=50, T=0, PV=100.0
F=90, T=0, PV=100.0
F=100, T=0, PV=100.0
F=110, T=0, PV=100.0
F=150, T=0, PV=100.0
r=0.1, implied box rate = 0.10000000000000069


In [122]:
# Compute PV and financing P&L
pv_ts = dict()
day_to_maturity = list(range(0, nb_days+1))[::-1]

for i in day_to_maturity:
    T = i / 252
    pv = compute_box_pv(F=100, T=T, r=r)
    pv_ts[i] = pv
pv_ts = pd.Series(pv_ts)
pv_pnl = pv_ts.diff().fillna(0)
financing_pnl = pv_ts.shift().fillna(0) * (np.exp(r/252)-1)
total_pnl = (pv_pnl-financing_pnl).sum()
print(f'pv_pnl={pv_pnl.sum():.2F}, financing_pnl={financing_pnl.sum():.4F}, total_pnl={total_pnl:.4F}')

pv_pnl=9.52, financing_pnl=9.5163, total_pnl=-0.0000


## Forward Analysis

In [123]:

pv_ts = dict()
discount = dict()
forward_ts = dict()
for i in day_to_maturity:
    T = i / 252
    if i == nb_days:
        S = 100
        K_fwd = BSMModel.compute_forward(S=S, T=252/252, r=r, g=0, q=0)
    else:
        S = 100
    forward_ts[i] = BSMModel.compute_forward(S=S, T=T, r=r, g=0, q=0)
    a = BSMModel.compute_option_with_forward(F=forward_ts[i], K=K_fwd, T=T, r=r, sigma=0.2, option_type='call')
    b = BSMModel.compute_option_with_forward(F=forward_ts[i], K=K_fwd, T=T, r=r, sigma=0.2, option_type='put')
    pv_ts[i] = a-b
    discount[i] = np.exp(-r*T)
discount = pd.Series(discount)

In [124]:
fwd_pv = (pd.Series(forward_ts)-K_fwd) * discount
pv_ts = pd.Series(pv_ts)
pv_diff = (pv_ts-fwd_pv).sum()
print(f'synth fwd (option) pv - foward pv : {pv_diff:.4F}')

synth fwd (option) pv - foward pv : -0.0000


In [125]:
pv_pnl = pv_ts.diff().fillna(0)
financing_pnl = pv_ts.shift().fillna(0) * (np.exp(r/252)-1)
total_pnl = (pv_pnl - financing_pnl).sum()
print(f'pv_pnl={pv_pnl.sum():.2F}, financing_pnl={financing_pnl.sum():.4F}, total_pnl={total_pnl:.4F}')

pv_pnl=-10.52, financing_pnl=-0.5151, total_pnl=-10.0020
